In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

from utils import (
    get_hdfs_base_uri,
    create_namespace,
    add_hash_key,
    add_record_hash,
    add_gold_metadata,
    validate_expected_columns,
    get_latest_batch_id,
    merge_iceberg_by_keys,
    replace_iceberg_by_partition_values,
    default_spark_local_dir,
    is_truthy,
)

SPARK_LOCAL_DIR = default_spark_local_dir()
os.environ["SPARK_LOCAL_DIRS"] = SPARK_LOCAL_DIR
os.makedirs(SPARK_LOCAL_DIR, exist_ok=True)

spark = (
    SparkSession.builder
    .config("spark.local.dir", SPARK_LOCAL_DIR)
    .getOrCreate()
)

# ============================================================
# Configurações
# ============================================================

HDFS_BASE_URI = get_hdfs_base_uri()

CATALOG = "spark_catalog"

SILVER_DB = "silver"
GOLD_DB = "gold"

SILVER_OPERADORA_TABLE = f"{CATALOG}.{SILVER_DB}.operadora"
SILVER_MUNICIPIO_TABLE = f"{CATALOG}.{SILVER_DB}.municipio"
SILVER_PLANO_TABLE = f"{CATALOG}.{SILVER_DB}.plano"
SILVER_MOVIMENTO_TABLE = f"{CATALOG}.{SILVER_DB}.beneficiario_movimento"

GOLD_DIM_OPERADORA_TABLE = f"{CATALOG}.{GOLD_DB}.dim_operadora"
GOLD_DIM_MUNICIPIO_TABLE = f"{CATALOG}.{GOLD_DB}.dim_municipio"
GOLD_DIM_PLANO_TABLE = f"{CATALOG}.{GOLD_DB}.dim_plano"
GOLD_DIM_PERFIL_TABLE = f"{CATALOG}.{GOLD_DB}.dim_perfil_beneficiario"
GOLD_FATO_MOVIMENTO_TABLE = f"{CATALOG}.{GOLD_DB}.fato_beneficiario_movimento"

GOLD_LOCATION = f"{HDFS_BASE_URI}/user/hive/warehouse/{GOLD_DB}.db"

FULL_SNAPSHOT_REPLACE = is_truthy(os.getenv("FULL_SNAPSHOT_REPLACE", "true"))

create_namespace(
    spark=spark,
    catalog=CATALOG,
    database=GOLD_DB,
    location=GOLD_LOCATION,
)

BATCH_ID = os.getenv("BATCH_ID")

if not BATCH_ID:
    BATCH_ID = get_latest_batch_id(
        spark=spark,
        table_name=SILVER_MOVIMENTO_TABLE,
        batch_column="_batch_id",
    )

print(f"Processando camada Gold para o batch: {BATCH_ID}")

# ============================================================
# Leitura das tabelas Silver
# ============================================================

silver_movimento = (
    spark.table(SILVER_MOVIMENTO_TABLE)
    .where(F.col("_batch_id") == BATCH_ID)
)

silver_operadora = (
    spark.table(SILVER_OPERADORA_TABLE)
    .where(F.col("_batch_id") == BATCH_ID)
)

silver_municipio = (
    spark.table(SILVER_MUNICIPIO_TABLE)
    .where(F.col("_batch_id") == BATCH_ID)
)

silver_plano = (
    spark.table(SILVER_PLANO_TABLE)
    .where(F.col("_batch_id") == BATCH_ID)
)

# ============================================================
# Validação mínima das colunas usadas na Gold
# ============================================================

EXPECTED_MOVIMENTO_COLUMNS = [
    "id_cmpt_movel",
    "cd_operadora",
    "cd_municipio",
    "cd_plano",
    "tp_sexo",
    "de_faixa_etaria",
    "de_faixa_etaria_reaj",
    "tipo_vinculo",
    "qt_beneficiario_ativo",
    "qt_beneficiario_aderido",
    "qt_beneficiario_cancelado",
]

EXPECTED_OPERADORA_COLUMNS = [
    "cd_operadora",
    "nm_razao_social",
    "nr_cnpj",
    "modalidade_operadora",
]

EXPECTED_MUNICIPIO_COLUMNS = [
    "cd_municipio",
    "nm_municipio",
    "sg_uf",
]

EXPECTED_PLANO_COLUMNS = [
    "cd_operadora",
    "cd_plano",
    "tp_vigencia_plano",
    "de_contratacao_plano",
    "de_segmentacao_plano",
    "de_abrg_geografica_plano",
    "cobertura_assist_plan",
]

validate_expected_columns(silver_movimento, EXPECTED_MOVIMENTO_COLUMNS)
validate_expected_columns(silver_operadora, EXPECTED_OPERADORA_COLUMNS)
validate_expected_columns(silver_municipio, EXPECTED_MUNICIPIO_COLUMNS)
validate_expected_columns(silver_plano, EXPECTED_PLANO_COLUMNS)

if silver_movimento.limit(1).count() == 0:
    raise ValueError(f"Nenhum registro encontrado na Silver para o batch {BATCH_ID}.")

# ============================================================
# Dimensão: Operadora
# Grão: 1 linha por cd_operadora
# ============================================================

df_dim_operadora = (
    silver_operadora
    .select(
        "cd_operadora",
        "nm_razao_social",
        "nr_cnpj",
        "modalidade_operadora",
    )
    .where(F.col("cd_operadora").isNotNull())
    .dropDuplicates(["cd_operadora"])
)

df_dim_operadora = add_hash_key(
    df=df_dim_operadora,
    key_columns=["cd_operadora"],
    key_column_name="sk_operadora",
    key_namespace="dim_operadora",
)

df_dim_operadora = add_record_hash(
    df=df_dim_operadora,
    payload_columns=[
        "sk_operadora",
        "cd_operadora",
        "nm_razao_social",
        "nr_cnpj",
        "modalidade_operadora",
    ],
    hash_column_name="_record_hash",
)

df_dim_operadora = (
    add_gold_metadata(df_dim_operadora, BATCH_ID)
    .select(
        "sk_operadora",
        "cd_operadora",
        "nm_razao_social",
        "nr_cnpj",
        "modalidade_operadora",
        "_record_hash",
        "_batch_id",
        "_gold_ingested_at",
        "_layer",
    )
)

# ============================================================
# Dimensão: Município
# Grão: 1 linha por cd_municipio
# ============================================================

df_dim_municipio = (
    silver_municipio
    .select(
        "cd_municipio",
        "nm_municipio",
        "sg_uf",
    )
    .where(F.col("cd_municipio").isNotNull())
    .dropDuplicates(["cd_municipio"])
)

df_dim_municipio = add_hash_key(
    df=df_dim_municipio,
    key_columns=["cd_municipio"],
    key_column_name="sk_municipio",
    key_namespace="dim_municipio",
)

df_dim_municipio = add_record_hash(
    df=df_dim_municipio,
    payload_columns=[
        "sk_municipio",
        "cd_municipio",
        "nm_municipio",
        "sg_uf",
    ],
    hash_column_name="_record_hash",
)

df_dim_municipio = (
    add_gold_metadata(df_dim_municipio, BATCH_ID)
    .select(
        "sk_municipio",
        "cd_municipio",
        "nm_municipio",
        "sg_uf",
        "_record_hash",
        "_batch_id",
        "_gold_ingested_at",
        "_layer",
    )
)

# ============================================================
# Dimensão: Plano
# Grão: 1 linha por cd_operadora + cd_plano
# ============================================================

df_dim_plano = (
    silver_plano
    .select(
        "cd_operadora",
        "cd_plano",
        "tp_vigencia_plano",
        "de_contratacao_plano",
        "de_segmentacao_plano",
        "de_abrg_geografica_plano",
        "cobertura_assist_plan",
    )
    .where(
        F.col("cd_operadora").isNotNull()
        & F.col("cd_plano").isNotNull()
    )
    .dropDuplicates(["cd_operadora", "cd_plano"])
)

df_dim_plano = add_hash_key(
    df=df_dim_plano,
    key_columns=["cd_operadora", "cd_plano"],
    key_column_name="sk_plano",
    key_namespace="dim_plano",
)

df_dim_plano = add_record_hash(
    df=df_dim_plano,
    payload_columns=[
        "sk_plano",
        "cd_operadora",
        "cd_plano",
        "tp_vigencia_plano",
        "de_contratacao_plano",
        "de_segmentacao_plano",
        "de_abrg_geografica_plano",
        "cobertura_assist_plan",
    ],
    hash_column_name="_record_hash",
)

df_dim_plano = (
    add_gold_metadata(df_dim_plano, BATCH_ID)
    .select(
        "sk_plano",
        "cd_operadora",
        "cd_plano",
        "tp_vigencia_plano",
        "de_contratacao_plano",
        "de_segmentacao_plano",
        "de_abrg_geografica_plano",
        "cobertura_assist_plan",
        "_record_hash",
        "_batch_id",
        "_gold_ingested_at",
        "_layer",
    )
)

# ============================================================
# Dimensão: Perfil do beneficiário
# Grão: 1 linha por sexo + faixa etária + faixa reajuste + tipo vínculo
# ============================================================

df_dim_perfil = (
    silver_movimento
    .select(
        "tp_sexo",
        "de_faixa_etaria",
        "de_faixa_etaria_reaj",
        "tipo_vinculo",
    )
    .where(
        F.col("tp_sexo").isNotNull()
        & F.col("de_faixa_etaria").isNotNull()
        & F.col("de_faixa_etaria_reaj").isNotNull()
        & F.col("tipo_vinculo").isNotNull()
    )
    .distinct()
)

df_dim_perfil = add_hash_key(
    df=df_dim_perfil,
    key_columns=[
        "tp_sexo",
        "de_faixa_etaria",
        "de_faixa_etaria_reaj",
        "tipo_vinculo",
    ],
    key_column_name="sk_perfil_beneficiario",
    key_namespace="dim_perfil_beneficiario",
)

df_dim_perfil = add_record_hash(
    df=df_dim_perfil,
    payload_columns=[
        "sk_perfil_beneficiario",
        "tp_sexo",
        "de_faixa_etaria",
        "de_faixa_etaria_reaj",
        "tipo_vinculo",
    ],
    hash_column_name="_record_hash",
)

df_dim_perfil = (
    add_gold_metadata(df_dim_perfil, BATCH_ID)
    .select(
        "sk_perfil_beneficiario",
        "tp_sexo",
        "de_faixa_etaria",
        "de_faixa_etaria_reaj",
        "tipo_vinculo",
        "_record_hash",
        "_batch_id",
        "_gold_ingested_at",
        "_layer",
    )
)

# ============================================================
# Escrita das dimensões
# ============================================================

merge_iceberg_by_keys(
    spark=spark,
    source_df=df_dim_operadora,
    target_table=GOLD_DIM_OPERADORA_TABLE,
    key_columns=["sk_operadora"],
    source_view="vw_gold_dim_operadora",
    compare_hash_column="_record_hash",
)

merge_iceberg_by_keys(
    spark=spark,
    source_df=df_dim_municipio,
    target_table=GOLD_DIM_MUNICIPIO_TABLE,
    key_columns=["sk_municipio"],
    source_view="vw_gold_dim_municipio",
    compare_hash_column="_record_hash",
)

merge_iceberg_by_keys(
    spark=spark,
    source_df=df_dim_plano,
    target_table=GOLD_DIM_PLANO_TABLE,
    key_columns=["sk_plano"],
    source_view="vw_gold_dim_plano",
    compare_hash_column="_record_hash",
)

merge_iceberg_by_keys(
    spark=spark,
    source_df=df_dim_perfil,
    target_table=GOLD_DIM_PERFIL_TABLE,
    key_columns=["sk_perfil_beneficiario"],
    source_view="vw_gold_dim_perfil_beneficiario",
    compare_hash_column="_record_hash",
)

# ============================================================
# Fato: Movimento de beneficiários
# Grão: competência + operadora + município + plano + perfil
# ============================================================

df_fato = silver_movimento

df_fato = (
    df_fato
    .withColumn("ano_competencia", F.substring(F.col("id_cmpt_movel"), 1, 4).cast("int"))
    .withColumn("mes_competencia", F.substring(F.col("id_cmpt_movel"), 5, 2).cast("int"))
    .withColumn(
        "dt_competencia",
        F.to_date(F.concat(F.col("id_cmpt_movel"), F.lit("01")), "yyyyMMdd"),
    )
    .withColumn("ds_competencia", F.date_format(F.col("dt_competencia"), "yyyy-MM"))
)

df_fato = add_hash_key(
    df=df_fato,
    key_columns=["cd_operadora"],
    key_column_name="sk_operadora",
    key_namespace="dim_operadora",
)

df_fato = add_hash_key(
    df=df_fato,
    key_columns=["cd_municipio"],
    key_column_name="sk_municipio",
    key_namespace="dim_municipio",
)

df_fato = add_hash_key(
    df=df_fato,
    key_columns=["cd_operadora", "cd_plano"],
    key_column_name="sk_plano",
    key_namespace="dim_plano",
)

df_fato = add_hash_key(
    df=df_fato,
    key_columns=[
        "tp_sexo",
        "de_faixa_etaria",
        "de_faixa_etaria_reaj",
        "tipo_vinculo",
    ],
    key_column_name="sk_perfil_beneficiario",
    key_namespace="dim_perfil_beneficiario",
)

df_fato = (
    df_fato
    .select(
        "id_cmpt_movel",
        "ano_competencia",
        "mes_competencia",
        "dt_competencia",
        "ds_competencia",
        "sk_operadora",
        "sk_municipio",
        "sk_plano",
        "sk_perfil_beneficiario",
        "qt_beneficiario_ativo",
        "qt_beneficiario_aderido",
        "qt_beneficiario_cancelado",
    )
)

df_fato = add_record_hash(
    df=df_fato,
    payload_columns=[
        "id_cmpt_movel",
        "ano_competencia",
        "mes_competencia",
        "dt_competencia",
        "ds_competencia",
        "sk_operadora",
        "sk_municipio",
        "sk_plano",
        "sk_perfil_beneficiario",
        "qt_beneficiario_ativo",
        "qt_beneficiario_aderido",
        "qt_beneficiario_cancelado",
    ],
    hash_column_name="_record_hash",
)

df_fato = (
    add_gold_metadata(df_fato, BATCH_ID)
    .select(
        "id_cmpt_movel",
        "ano_competencia",
        "mes_competencia",
        "dt_competencia",
        "ds_competencia",
        "sk_operadora",
        "sk_municipio",
        "sk_plano",
        "sk_perfil_beneficiario",
        "qt_beneficiario_ativo",
        "qt_beneficiario_aderido",
        "qt_beneficiario_cancelado",
        "_record_hash",
        "_batch_id",
        "_gold_ingested_at",
        "_layer",
    )
)

# ============================================================
# Escrita da fato
# ============================================================

FACT_KEY_COLUMNS = [
    "id_cmpt_movel",
    "sk_operadora",
    "sk_municipio",
    "sk_plano",
    "sk_perfil_beneficiario",
]

if FULL_SNAPSHOT_REPLACE:
    replace_iceberg_by_partition_values(
        spark=spark,
        source_df=df_fato,
        target_table=GOLD_FATO_MOVIMENTO_TABLE,
        partition_columns=["id_cmpt_movel"],
        source_view="vw_gold_fato_movimento_snapshot",
        keys_view="vw_gold_fato_movimento_snapshot_keys",
    )
else:
    merge_iceberg_by_keys(
        spark=spark,
        source_df=df_fato,
        target_table=GOLD_FATO_MOVIMENTO_TABLE,
        key_columns=FACT_KEY_COLUMNS,
        source_view="vw_gold_fato_movimento_incremental",
        compare_hash_column="_record_hash",
    )